In [123]:
import pandas as pd
import numpy as np

btc = pd.read_csv('data/btc_daily.csv', index_col=0)
btc.index = pd.to_datetime(btc.index)
btc.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
btc = btc[-10000:]
btc

,Open,High,Low,Close,Volume
2015-07-20,277.98,280.00,277.37,280.00,782.883420
2015-07-21,279.96,281.27,276.85,277.32,4943.559434
2015-07-22,277.33,278.54,275.01,277.89,4687.909383
2015-07-23,277.96,279.75,276.28,277.39,5306.919575
2015-07-24,277.23,291.52,276.43,289.12,7362.469083
...,...,...,...,...,...
2025-03-23,83852.06,86128.99,83804.67,86092.94,3467.469453
2025-03-24,86092.95,88804.64,85533.06,87523.62,13107.503916
2025-03-25,87518.09,88584.34,86321.97,87427.88,10153.983124
2025-03-26,87432.82,88304.10,85850.00,86926.01,6885.719860


Parameters:
- Lookback_period = 20 (days)
- body_multiplier = 1.5
- back_candles = 50
- test_candles = 10

# Detect FVG

In [124]:
def detect_fvg(data, lookback_period=10, body_multiplier=1.5):
    """
    Detects Fair Value Gaps (FVGs) in historical price data.

    Parameters:
        data (DataFrame): DataFrame with columns ['open', 'high', 'low', 'close'].
        lookback_period (int): Number of candles to look back for average body size.
        body_multiplier (float): Multiplier to determine significant body size.

    Returns:
        list of tuples: Each tuple contains ('type', start, end, index).
    """
    fvg_list = [None, None]

    for i in range(2, len(data)):
        first_high = data['High'].iloc[i-2]
        first_low = data['Low'].iloc[i-2]
        middle_open = data['Open'].iloc[i-1]
        middle_close = data['Close'].iloc[i-1]
        third_low = data['Low'].iloc[i]
        third_high = data['High'].iloc[i]

        # Calculate the average absolute body size over the lookback period
        prev_bodies = (data['Close'].iloc[max(0, i-1-lookback_period):i-1] - 
                       data['Open'].iloc[max(0, i-1-lookback_period):i-1]).abs()
        avg_body_size = prev_bodies.mean()
        
        # Ensure avg_body_size is nonzero to avoid false positives
        avg_body_size = avg_body_size if avg_body_size > 0 else 0.001

        middle_body = abs(middle_close - middle_open)

        # Check for Bullish FVG
        if third_low > first_high and middle_body > avg_body_size * body_multiplier:
            fvg_list.append(('bullish', first_high, third_low, i))

        # Check for Bearish FVG
        elif third_high < first_low and middle_body > avg_body_size * body_multiplier:
            fvg_list.append(('bearish', first_low, third_high, i))
        else:
            fvg_list.append(None)

    return fvg_list



In [ ]:
# need to play around with parameters
btc['FVG'] = detect_fvg(btc, lookback_period=30, body_multiplier=1)
print(len(btc[~btc['FVG'].isna()]))
btc

627


,Open,High,Low,Close,Volume,FVG
2015-07-20,277.98,280.00,277.37,280.00,782.883420,None
2015-07-21,279.96,281.27,276.85,277.32,4943.559434,None
2015-07-22,277.33,278.54,275.01,277.89,4687.909383,None
2015-07-23,277.96,279.75,276.28,277.39,5306.919575,None
2015-07-24,277.23,291.52,276.43,289.12,7362.469083,None
...,...,...,...,...,...,...
2025-03-23,83852.06,86128.99,83804.67,86092.94,3467.469453,None
2025-03-24,86092.95,88804.64,85533.06,87523.62,13107.503916,None
2025-03-25,87518.09,88584.34,86321.97,87427.88,10153.983124,None
2025-03-26,87432.82,88304.10,85850.00,86926.01,6885.719860,None


# Visualise FVG

In [126]:
import plotly.graph_objects as go
from datetime import datetime

dfpl = btc[-1000:]
# Create the figure
fig = go.Figure()

# Add candlestick chart
fig.add_trace(go.Candlestick(
    x=dfpl.index,
    open=dfpl["Open"],
    high=dfpl["High"],
    low=dfpl["Low"],
    close=dfpl["Close"],
    name="Candles"
))

# Add FVG zones
for _, row in dfpl.iterrows():
    if isinstance(row["FVG"], tuple):
        fvg_type, start, end, index = row["FVG"]
        color = "rgba(0,255,0,0.3)" if fvg_type == "bullish" else "rgba(255,0,0,0.3)"
        current_date = row.name
        time_delta = (dfpl.index[1] - dfpl.index[0]) * 2
        forward_delta = (dfpl.index[1] - dfpl.index[0]) * 50
        fig.add_shape(
            type="rect",
            x0=current_date - time_delta,
            x1=current_date + forward_delta,
            y0=start,
            y1=end,
            fillcolor=color,
            opacity=0.8,
            layer="below",
            line=dict(width=0),
        )

# Show the chart
fig.update_layout(width=1200, height=800,
                  xaxis=dict(showgrid=False),
                  yaxis=dict(showgrid=False),
                  plot_bgcolor='black',
                  paper_bgcolor='black')
fig.show()

Interpretation:
- Candlestick Chart: The main display shows Daily Bitcoin price movements with candlesticks - green candles represent price increases, while red candles show price decreases.
- Price Range: The y-axis shows the Bitcoin price.
- Colored Rectangles: Fair Value Gaps (FVGs)
    - Green rectangles: Bullish FVGs (where price moved up quickly, creating a gap)
    - Red rectangles: Bearish FVGs (where price moved down quickly, creating a gap)


Intuition:
- Green rectangles (Bullish FVGs): These indicate potential buying opportunities, especially if price returns to test this level from above AND the previous candle breaks above a key resistance level.
- Red rectangles (Bearish FVGs): These indicate potential selling opportunities in, especially if price returns to test this level from below AND the previous candle breaks below a key support level.


Note:
- FVG on its own is not enough. We need something else to reinforce these claims

# Key Levels

In [127]:
def detect_key_levels(df, current_candle, backcandles=50, test_candles=10):
    """
    Detects key support and resistance levels in a given backcandles window.
    
    A level is identified if a candle's high is the highest or its low is the lowest 
    compared to `test_candles` before and after it.

    Parameters:
        df (pd.DataFrame): DataFrame containing 'High' and 'Low' columns.
        current_candle (int): The index of the current candle (latest available candle).
        backcandles (int): Number of candles to look back.
        test_candles (int): Number of candles before and after each candle to check.

    Returns:
        dict: A dictionary with detected 'support' and 'resistance' levels.
    """
    key_levels = {"support": [], "resistance": []}

    # Define the last candle that can be tested to avoid lookahead bias
    last_testable_candle = current_candle - test_candles

    # Ensure we have enough data
    if last_testable_candle < backcandles + test_candles:
        return key_levels  # Not enough historical data

    # Iterate through the backcandles window
    for i in range(current_candle - backcandles, last_testable_candle):
        high = df['High'].iloc[i]
        low = df['Low'].iloc[i]

        # Get surrounding window of test_candles before and after
        before = df.iloc[max(0, i - test_candles):i]
        after = df.iloc[i + 1: min(len(df), i + test_candles + 1)]

        # Check if current high is the highest among before & after candles
        if high > before['High'].max() and high > after['High'].max():
            key_levels["resistance"].append((i, high))

        # Check if current low is the lowest among before & after candles
        if low < before['Low'].min() and low < after['Low'].min():
            key_levels["support"].append((i, low))

    return key_levels

def fill_key_levels(df, backcandles=50, test_candles=10):
    """
    Adds a 'key_levels' column to the DataFrame where each row contains all
    key support and resistance levels detected up to that candle (including
    both the level value and the index of the candle that generated it).
    
    Parameters:
        df (pd.DataFrame): DataFrame containing 'High' and 'Low' columns.
        backcandles (int): Lookback window for detecting key levels.
        test_candles (int): Number of candles before/after for validation.

    Returns:
        pd.DataFrame: Updated DataFrame with the new 'key_levels' column.
    """
    df["key_levels"] = None  # Initialize the column
    
    from tqdm import tqdm
    for current_candle in tqdm(range(backcandles + test_candles, len(df))):
        # Detect key levels for the current candle
        key_levels = detect_key_levels(df, current_candle, backcandles, test_candles)

        # Collect support and resistance levels (with their indices) up to current_candle
        support_levels = [(idx, level) for (idx, level) in key_levels["support"] 
                          if idx < current_candle]
        resistance_levels = [(idx, level) for (idx, level) in key_levels["resistance"] 
                             if idx < current_candle]

        # Store the levels along with the originating candle index
        if support_levels or resistance_levels:
            df.at[current_candle, "key_levels"] = {
                "support": support_levels,
                "resistance": resistance_levels
            }
            
    return df

btc = fill_key_levels(btc.reset_index(), backcandles=50, test_candles=10)
# btc.set_index('index', drop=True)
btc

100%|██████████| 3479/3479 [00:09<00:00, 381.18it/s]


,index,Open,High,Low,Close,Volume,FVG,key_levels
0,2015-07-20,277.98,280.00,277.37,280.00,782.883420,None,None
1,2015-07-21,279.96,281.27,276.85,277.32,4943.559434,None,None
2,2015-07-22,277.33,278.54,275.01,277.89,4687.909383,None,None
3,2015-07-23,277.96,279.75,276.28,277.39,5306.919575,None,None
4,2015-07-24,277.23,291.52,276.43,289.12,7362.469083,None,None
...,...,...,...,...,...,...,...,...
3534,2025-03-23,83852.06,86128.99,83804.67,86092.94,3467.469453,None,"{'support': [(3486, 91178.01), (3522, 76555.0)..."
3535,2025-03-24,86092.95,88804.64,85533.06,87523.62,13107.503916,None,"{'support': [(3486, 91178.01), (3522, 76555.0)..."
3536,2025-03-25,87518.09,88584.34,86321.97,87427.88,10153.983124,None,"{'support': [(3486, 91178.01), (3522, 76555.0)..."
3537,2025-03-26,87432.82,88304.10,85850.00,86926.01,6885.719860,None,"{'support': [(3522, 76555.0)], 'resistance': [..."


# Visualize Key Levels

In [128]:

def plot_fvg_and_key_levels(df, start_idx, end_idx, extension=30):
    """
    Plots candlesticks, FVG zones, and key levels (support/resistance) for a
    subset of a DataFrame from `start_idx` to `end_idx`.
    
    The FVG column is expected to have tuples of the form:
        (fvg_type, start_price, end_price, trigger_index)

    The key_levels column is expected to have dictionaries of the form:
        {
          "support": [(idx, price), (idx, price), ...],
          "resistance": [(idx, price), (idx, price), ...]
        }

    Parameters:
    -----------
    df : pd.DataFrame
        Must contain: "Open", "High", "Low", "Close", "FVG", "key_levels".
    start_idx : int
        Starting row index for plotting.
    end_idx : int
        Ending row index for plotting.
    extension : int
        How far (in x-axis units/index steps) to extend the FVG rectangles
        and key-level lines.
    
    Returns:
    --------
    fig : plotly.graph_objects.Figure
        A Plotly Figure with the candlesticks, FVG, and key-level lines.
    """
    
    # Slice the DataFrame to the desired plotting range
    dfpl = df.loc[start_idx:end_idx]

    # Create the figure
    fig = go.Figure()

    # -- 1) Add Candlestick Chart --
    fig.add_trace(go.Candlestick(
        x=dfpl.index,
        open=dfpl["Open"],
        high=dfpl["High"],
        low=dfpl["Low"],
        close=dfpl["Close"],
        name="Candles"
    ))

    # -- 2) Add FVG Zones --
    for i, row in dfpl.iterrows():
        # Check if "FVG" is a valid tuple: (fvg_type, start_price, end_price, trigger_index)
        if isinstance(row.get("FVG"), tuple):
            fvg_type, start_price, end_price, trigger_idx = row["FVG"]

            # Choose a fill color based on bullish vs. bearish
            if fvg_type == "bullish":
                color = "rgba(0, 255, 0, 0.3)"   # greenish
            else:
                color = "rgba(255, 0, 0, 0.3)"   # reddish

            fig.add_shape(
                type="rect",
                x0=trigger_idx, 
                x1=trigger_idx + extension,
                y0=start_price,
                y1=end_price,
                fillcolor=color,
                opacity=0.4,
                layer="below",
                line=dict(width=0),
            )

    # -- 3) Add Key Levels as Horizontal Lines --
    for i, row in dfpl.iterrows():
        key_levels = row.get("key_levels", None)
        if key_levels:
            # key_levels is a dict: {"support": [(idx, val), ...], "resistance": [(idx, val), ...]}
            support_levels = key_levels.get("support", [])
            resistance_levels = key_levels.get("resistance", [])

            # Plot support levels
            for (gen_idx, s_price) in support_levels:
                # We only draw the line if gen_idx is in (start_idx, end_idx)
                # You can decide to relax/omit this check if you want lines from outside the window.
                if start_idx <= gen_idx <= end_idx:
                    fig.add_shape(
                        type="line",
                        x0=gen_idx,
                        x1=gen_idx + extension,
                        y0=s_price,
                        y1=s_price,
                        line=dict(color="blue", width=2),
                        layer="below"
                    )

            # Plot resistance levels
            for (gen_idx, r_price) in resistance_levels:
                if start_idx <= gen_idx <= end_idx:
                    fig.add_shape(
                        type="line",
                        x0=gen_idx,
                        x1=gen_idx + extension,
                        y0=r_price,
                        y1=r_price,
                        line=dict(color="orange", width=2),
                        layer="below"
                    )

    # -- 4) Figure Aesthetics --
    fig.update_layout(
        width=1200,
        height=800,
        xaxis=dict(showgrid=False),
        yaxis=dict(showgrid=False),
        plot_bgcolor='black',
        paper_bgcolor='black'
    )
    return fig

fig = plot_fvg_and_key_levels(btc, start_idx=len(btc)-201, end_idx=len(btc)-1, extension=50)
fig.show()

# Detect the Signal

In [129]:
def detect_break_signal(df):
    """
    Detects if the current candle carries an FVG signal and,
    at the same time, the previous candle has crossed a key level
    in the expected direction (up for bullish, down for bearish).

    - If FVG is bullish and previous candle crosses ABOVE a level -> signal = 2
    - If FVG is bearish and previous candle crosses BELOW a level -> signal = 1
    - Otherwise -> signal = 0

    The 'FVG' column is expected to have tuples like:
        (fvg_type, lower_price, upper_price, trigger_index)
      where fvg_type is either "bullish" or "bearish".

    The 'key_levels' column is expected to be a dictionary with:
        {
            'support': [(level_candle_idx, level_price), ...],
            'resistance': [(level_candle_idx, level_price), ...]
        }
    """

    # Initialize the new signal column to 0
    df["break_signal"] = 0

    # We start at 1 because we compare candle i with its previous candle (i-1)
    for i in range(1, len(df)):
        fvg = df.loc[i, "FVG"]
        key_levels = df.loc[i, "key_levels"]

        # We only proceed if there's an FVG tuple and some key_levels dict
        if isinstance(fvg, tuple) and isinstance(key_levels, dict):
            fvg_type = fvg[0]  # "bullish" or "bearish"

            # Previous candle's OHLC
            prev_open = df.loc[i-1, "Open"]
            prev_close = df.loc[i-1, "Close"]

            # -----------------------
            # 1) Bullish FVG check
            # -----------------------
            if fvg_type == "bullish":
                # Typically you'd check crossing a "resistance" level
                # crossing means the previous candle goes from below -> above
                resistance_levels = key_levels.get("resistance", [])
                
                for (lvl_idx, lvl_price) in resistance_levels:
                    # Condition: previously below, ended above
                    # simplest check is: prev_open < lvl_price < prev_close
                    if prev_open < lvl_price and prev_close > lvl_price:
                        df.loc[i, "break_signal"] = 2
                        break  # No need to check more levels

            # -----------------------
            # 2) Bearish FVG check
            # -----------------------
            elif fvg_type == "bearish":
                # Typically you'd check crossing a "support" level
                support_levels = key_levels.get("support", [])
                
                for (lvl_idx, lvl_price) in support_levels:
                    # Condition: previously above, ended below
                    # simplest check is: prev_open > lvl_price and prev_close < lvl_price
                    if prev_open > lvl_price and prev_close < lvl_price:
                        df.loc[i, "break_signal"] = 1
                        break  # No need to check more levels

    return df

btc = detect_break_signal(btc)
btc[btc["break_signal"]!=0]

,index,Open,High,Low,Close,Volume,FVG,key_levels,break_signal
86,2015-10-14,250.98,255.00,250.25,253.27,7587.526674,"(bullish, 249.14, 250.25, 86)","{'support': [(36, 198.02), (64, 224.45)], 'res...",2
180,2016-01-16,357.59,391.00,350.92,388.70,17985.238784,"(bearish, 428.0, 391.0, 180)","{'support': [(159, 410.0)], 'resistance': [(14...",1
271,2016-04-16,430.82,434.89,429.89,433.39,4133.985987,"(bullish, 428.0, 429.89, 271)","{'support': [(229, 381.09), (242, 399.21)], 'r...",2
313,2016-05-28,471.21,539.49,470.13,523.25,12107.456597,"(bullish, 454.99, 470.13, 313)","{'support': [], 'resistance': [(280, 472.38), ...",2
314,2016-05-29,523.35,549.99,491.01,525.22,9957.527243,"(bullish, 476.75, 491.01, 314)","{'support': [], 'resistance': [(280, 472.38), ...",2
...,...,...,...,...,...,...,...,...,...
3208,2024-05-01,60621.20,60785.49,56500.00,58265.59,26611.931273,"(bearish, 61741.01, 60785.49, 3208)","{'support': [(3166, 60771.14), (3196, 59573.32...",1
3357,2024-09-27,65176.39,66550.00,64827.85,65789.00,9529.016881,"(bullish, 64811.0, 64827.85, 3357)","{'support': [(3336, 52530.0)], 'resistance': [...",2
3376,2024-10-16,67056.62,68399.00,66743.00,67608.44,12465.756912,"(bullish, 66500.0, 66743.0, 3376)","{'support': [(3336, 52530.0)], 'resistance': [...",2
3433,2024-12-12,101211.61,102595.00,99298.39,100030.47,14796.924565,"(bullish, 98338.17, 99298.39, 3433)","{'support': [], 'resistance': [(3413, 99860.0)]}",2


# Plot the Point position of the signal

In [132]:
def pointpos(x):
    if x['break_signal']==2:
        return x['Low']-1e-4
    elif x['break_signal']==1:
        return x['High']+1e-4
    else:
        return np.nan

btc['pointpos'] = btc.apply(lambda row: pointpos(row), axis=1)


strt = 2800
end = 3100
fig = plot_fvg_and_key_levels(btc, start_idx=strt, end_idx=end, extension=30)
fig.add_scatter(x=btc.index[strt:end], y=btc['pointpos'][strt:end], mode="markers",
                marker=dict(size=8, color="MediumPurple"),
                name="pivot")
fig.show()


# Backtesting

In [131]:
from backtesting import Strategy, Backtest

class MyStrat(Strategy):
    

SyntaxError: incomplete input (890814611.py, line 4)